In [1]:
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [3]:
with open("intentsnew.json", "r") as file:
    data = json.load(file)

print("Total Intents:", len(data["intents"]))

Total Intents: 41


In [4]:
patterns = []
tags = []

for intent in data["intents"]:
    tag = intent["tag"]

    for pattern in intent["patterns"]:
        patterns.append(pattern)
        tags.append(tag)

print("Total Patterns:", len(patterns))
print("Total Tags:", len(tags))

Total Patterns: 10248
Total Tags: 10248


In [5]:
encoder = LabelEncoder()

y = encoder.fit_transform(tags)

print("Number of Classes:", len(encoder.classes_))
print(encoder.classes_)

Number of Classes: 41
['Eligibility Criteria' 'FMCIII' 'Steps to complete the admission' 'about'
 'canteen' 'clubs' 'college intake' 'computerhod' 'course' 'creator'
 'cutoff' 'document' 'event' 'extchod' 'facilities' 'fees' 'floors'
 'goodbye' 'greeting' 'hostel' 'hours' 'infrastructure' 'ithod' 'library'
 'location' 'menu' 'name' 'number' 'placement' 'principal' 'ragging'
 'random' 'salutaion' 'scholarship' 'sports' 'swear' 'syllabus' 'task'
 'trust' 'uniform' 'vacation']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    patterns,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Size:", len(X_train))
print("Test Size:", len(X_test))

Train Size: 8198
Test Size: 2050


In [7]:
tokenizer = Tokenizer(oov_token="<OOV>")

tokenizer.fit_on_texts(X_train)

vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary Size:", vocab_size)

Vocabulary Size: 333


In [8]:
print(tokenizer.word_index)

{'<OOV>': 1, 'information': 2, 'about': 3, 'you': 4, 'can': 5, 'explain': 6, 'details': 7, 'tell': 8, 'me': 9, 'related': 10, 'question': 11, 'query': 12, 'need': 13, 'regarding': 14, 'help': 15, '22': 16, '28': 17, '24': 18, '5': 19, '25': 20, '21': 21, '27': 22, '16': 23, '2': 24, '14': 25, '17': 26, '6': 27, '10': 28, '23': 29, '12': 30, '18': 31, '15': 32, '8': 33, '4': 34, '26': 35, '11': 36, '3': 37, '9': 38, '1': 39, '7': 40, 'college': 41, '19': 42, '29': 43, '13': 44, '30': 45, '20': 46, 'the': 47, 'hostel': 48, 'name': 49, 'fees': 50, 'admission': 51, 'number': 52, 'cutoff': 53, 'to': 54, 'canteen': 55, 'library': 56, 'facilities': 57, 'ragging': 58, 'sports': 59, 'trust': 60, 'principal': 61, 'uniform': 62, 'syllabus': 63, 'fmciii': 64, 'clubs': 65, 'infrastructure': 66, 'floors': 67, 'event': 68, 'scholarship': 69, 'menu': 70, 'eligibility': 71, 'placement': 72, 'document': 73, 'steps': 74, 'hours': 75, 'criteria': 76, 'goodbye': 77, 'intake': 78, 'random': 79, 'vacation': 

In [9]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [11]:
max_length = max(len(seq) for seq in X_train_seq)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding="post"
)

print("Max Length:", max_length)
print("Classes:", len(encoder.classes_))

Max Length: 9
Classes: 41


In [12]:
from tensorflow.keras.utils import to_categorical

y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

print(y_train_cat.shape)
print(y_test_cat.shape)

(8198, 41)
(2050, 41)


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout

model = Sequential([
    
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
    ),

    GRU(128),

    Dropout(0.3),

    Dense(64, activation="relu"),

    Dense(41, activation="softmax")
])

model.summary()

C:\Users\prasa\OneDrive\Desktop\Data_Scientist\PHASE - II\tfenv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru (GRU)                            │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [15]:
history = model.fit(
    X_train_pad,
    y_train_cat,
    epochs=50,
    batch_size=8,
    validation_data=(X_test_pad, y_test_cat)
)

Epoch 1/50
1025/1025 ━━━━━━━━━━━━━━━━━━━━ 31s 19ms/step - accuracy: 0.2038 - loss: 2.4579 - val_accuracy: 0.4044 - val_loss: 1.5334
Epoch 2/50
1025/1025 ━━━━━━━━━━━━━━━━━━━━ 17s 16ms/step - accuracy: 0.6564 - loss: 1.0237 - val_accuracy: 0.7912 - val_loss: 0.6106
Epoch 3/50
1025/1025 ━━━━━━━━━━━━━━━━━━━━ 17s 17ms/step - accuracy: 0.9427 - loss: 0.3205 - val_accuracy: 0.9795 - val_loss: 0.1597
Epoch 4/50
1025/1025 ━━━━━━━━━━━━━━━━━━━━ 16s 16ms/step - accuracy: 0.9741 - loss: 0.1569 - val_accuracy: 0.9790 - val_loss: 0.1494
Epoch 5/50
1025/1025 ━━━━━━━━━━━━━━━━━━━━ 17s 16ms/step - accuracy: 0.9763 - loss: 0.1219 - val_accuracy: 0.9849 - val_loss: 0.1214
Epoch 6/50
1025/1025 ━━━━━━━━━━━━━━━━━━━━ 16s 16ms/step - accuracy: 0.9823 - loss: 0.0898 - val_accuracy: 0.9859 - val_loss: 0.1107
Epoch 7/50
1025/1025 ━━━━━━━━━━━━━━━━━━━━ 16s 16ms/step - accuracy: 0.9829 - loss: 0.0736 - val_accuracy: 0.9859 - val_loss: 0.1156
Epoch 8/50
1025/1025 ━━━━━━━━━━━━━━━━━━━━ 16s 16ms/step - accuracy: 0.9857 -

In [16]:
loss, accuracy = model.evaluate(X_test_pad, y_test_cat)

print("Test Accuracy:", accuracy)

65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9912 - loss: 0.1964
Test Accuracy: 0.9912195205688477


In [17]:
model.save("gru_chatbot_model.h5")

In [18]:
import pickle

with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

In [19]:
with open("label_encoder.pkl", "wb") as file:
    pickle.dump(encoder, file)

In [20]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [21]:
def predict_intent(text):
    
    sequence = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding="post"
    )

    prediction = model.predict(padded, verbose=0)

    confidence = np.max(prediction)

    predicted_class = np.argmax(prediction)

    tag = encoder.inverse_transform(
        [predicted_class]
    )[0]

    return tag, confidence

In [22]:
while True:

    msg = input("You: ")

    if msg.lower() == "quit":
        break

    tag, confidence = predict_intent(msg)

    print("Tag:", tag)
    print("Confidence:", confidence)

You:  hostel fee


Tag: hostel
Confidence: 1.0


You:  


Tag: greeting
Confidence: 0.9991117


You:  bye


Tag: goodbye
Confidence: 1.0


You:  canteen 


Tag: canteen
Confidence: 0.999998


You:  quit


In [23]:
import random

def get_response(tag):

    for intent in data["intents"]:

        if intent["tag"] == tag:

            return random.choice(intent["responses"])

    return "I don't understand."

In [24]:
while True:

    msg = input("You: ")

    if msg.lower() == "quit":
        break

    tag, confidence = predict_intent(msg)

    response = get_response(tag)

    print("Bot:", response)
    print("Confidence:", round(confidence, 4))

You:  quit
